In [3]:
import kagglehub as kh
data = kh.dataset_download('chetankv/dogs-cats-images')
print(data)

Using Colab cache for faster access to the 'dogs-cats-images' dataset.
/kaggle/input/dogs-cats-images


In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Resize((224, 224)),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
)
train_set = datasets.ImageFolder("/kaggle/input/dogs-cats-images/dog vs cat/dataset/training_set", transform=transform)
test_set = datasets.ImageFolder("/kaggle/input/dogs-cats-images/dog vs cat/dataset/test_set", transform=transform)
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool  = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.flatten = nn.Flatten()

        self._to_linear = None
        self._get_flat_size()

        self.fc1 = nn.Linear(self._to_linear, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 2)

    def _get_flat_size(self):
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 224, 224)
            x = self.pool(F.relu(self.conv1(dummy)))
            x = self.pool(F.relu(self.conv2(x)))
            self._to_linear = x.numel()

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x
model = Net()

In [22]:
import torch.optim as optim
from torch.utils.data import DataLoader
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

for epoch in range(15):  # 15 epochs(depends on how accurate you want it to be)
    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        inputs, labels = data
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'Epoch {epoch + 1}, Loss: {running_loss / len(train_loader)}')

# Evaluating the accuracy
correct = 0
total = 0
with torch.no_grad():
    for data in test_loader:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(f'Accuracy on test set: {100 * correct / total}%')

Epoch 1, Loss: 0.2986670093536377
Epoch 2, Loss: 0.19059105733036996
Epoch 3, Loss: 0.09580596183240414
Epoch 4, Loss: 0.07068168315291405
Epoch 5, Loss: 0.10008962623775006
Epoch 6, Loss: 0.10245680522918701
Epoch 7, Loss: 0.031572172936052086
Epoch 8, Loss: 0.0263045272808522
Epoch 9, Loss: 0.017604341291822492
Epoch 10, Loss: 0.010387995609431528
Epoch 11, Loss: 0.003446593198052142
Epoch 12, Loss: 0.0021897852492693344
Epoch 13, Loss: 0.0003475881780905183
Epoch 14, Loss: 0.00012049243102956097
Epoch 15, Loss: 8.929149294272066e-05
Accuracy on test set: 72.6%


In [13]:
torch.save(model.state_dict(), 'model.pth')

In [14]:
model_loaded = Net()
model_loaded.load_state_dict(torch.load('model.pth'))
model_loaded.eval()

Net(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=44944, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=2, bias=True)
)

In [23]:
from PIL import Image
from torchvision import transforms
import torch

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

img = Image.open('cat_picture.jpg').convert("RGB")
img_tensor = transform(img).unsqueeze(0)

model_loaded.eval()
with torch.no_grad():
    output = model_loaded(img_tensor)
    _, predicted = torch.max(output, 1)
    classes = ['cats', 'dogs']
    print(f'Predicted class: {classes[predicted.item()]}')

Predicted class: cats
